 # КР чётный вариант: Распознавание ПДн в русскоязычных текстах
 Комбинированный подход: BERT (ФИО: Фамилия, Имя) + регулярные выражения (структурированные атрибуты).
 Атрибуты: SURNAME, NAME, INN, PASSPORT, CREDIT_CARD, OGRNIP, IP, EMAIL, BIRTH_CERT, KLADR

In [1]:
# 0. Установка зависимостей
# ============================================================
import sys, subprocess

# гарантирует установку в ТО ЖЕ окружение, где запущен ноутбук
def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

_pip("--upgrade", "pip", "setuptools", "wheel")
_pip("transformers==4.46.3", "datasets", "evaluate", "seqeval", "faker", "torch", "accelerate", "corus")
print("✅ Установка зависимостей завершена. Перезапустите ядро (Kernel → Restart).")

✅ Установка зависимостей завершена. Перезапустите ядро (Kernel → Restart).


In [2]:
# 1. Импорты, конфигурация, константы
# ============================================================
import os, re, random, zipfile, shutil, builtins, urllib.request
from typing import Set

import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
)
from datasets import Dataset as HFDataset, DatasetDict, Features, Sequence, Value, ClassLabel
import evaluate
from faker import Faker

# ---- воспроизводимость ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- константы ----
BASE_MODEL = "DeepPavlov/rubert-base-cased"
FACT_RUEVAL_URL = "https://github.com/dialogue-evaluation/factRuEval-2016/archive/master.zip"
DATA_DIR = "./data"         # целевая папка установки архива
MODEL_OUT = "./rubert_pii"  # целевая папка установки модели

MAX_LEN = 256       # длина последовательности в токенах после токеницации
NUM_EPOCHS = 7      # количество полных проходов по обучающей выборке
BATCH_SIZE = 16     # сколько примеров модель обрабатывает за один шаг оптимизатора
LR = 2e-5           # скорость обучения - размер шага оптимизатора
WEIGHT_DECAY = 0.01 # штраф за большие веса, чтобы модель не запоминала обучающую выборку

# ---- атрибуты чётного варианта ----
ATTRS = [
    "SURNAME", "NAME",
    "INN", "PASSPORT", "CREDIT_CARD",
    "OGRNIP", "IP", "EMAIL", "BIRTH_CERT", "KLADR",
]
# BIO-схема: для каждого атрибута два тега (B — начало, I — продолжение), плюс O — «не сущность».
LABELS = ["O"] + [f"{p}-{a}" for a in ATTRS for p in ("B","I")]

# Прямое и обратное отображение строковых тегов в числовые id
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

# список кодов регионов РФ — первые две цифры серии паспорта и кода КЛАДР.
PASSPORT_REGION_WHITELIST: Set[int] = {
    1,3,4,5,7,8,10,11,12,14,15,17,18,19,20,21,22,24,25,26,27,28,29,
    32,33,34,35,36,37,38,40,41,42,43,44,45,46,47,48,49,50,51,52,53,
    54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,
    75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,
    96,97,98,99,
}

print("✅Конфигурация загружена. Устройство:", "cuda" if torch.cuda.is_available() else "cpu")
print(f"Атрибутов: {len(ATTRS)} | BIO-меток: {len(LABELS)}")

c:\Dev\PDn\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅Конфигурация загружена. Устройство: cpu
Атрибутов: 10 | BIO-меток: 21


In [3]:
# 2. Функции проверки контрольных разрядов
# ============================================================
def _digits_only(s: str) -> str:
    """Оставляет в строке только цифры — убирает пробелы, дефисы, точки, скобки."""
    return re.sub(r"\D", "", s)

def validate_inn(digits: str) -> bool:
    """Проверяет 12-значный ИНН физического лица по алгоритму ФНС."""

    # Две последние цифры ИНН — контрольные. Они вычисляются как взвешенная сумма
    # первых 10 и 11 цифр с фиксированными коэффициентами, взятая по модулю 11
    # и затем по модулю 10. Совпадение с фактическими цифрами означает корректный ИНН.

    d = _digits_only(digits)
    if len(d) != 12: return False
    nums = [int(c) for c in d]
    c11 = [7,2,4,10,3,5,9,4,6,8]    # коэффициенты первый контрольный разряд
    c12 = [3,7,2,4,10,3,5,9,4,6,8]  # коэффициенты второй контрольный разряд
    n11 = (sum(nums[i]*c11[i] for i in range(10)) % 11) % 10
    if n11 != nums[10]: return False
    n12 = (sum(nums[i]*c12[i] for i in range(11)) % 11) % 10
    return n12 == nums[11]

def validate_luhn(card_number: str) -> bool:
    """Проверяет 16-значный номер банковской карты по алгоритму Луна."""
    
    # каждая вторая цифра справа умножается на 2; если произведение > 9,
    # из него вычитается 9 (эквивалентно сумме цифр). Сумма всех цифр должна
    # делиться на 10 — это признак валидного номера.
    
    d = _digits_only(card_number)
    if len(d) != 16: return False
    total = 0
    for i, ch in enumerate(reversed(d)):
        n = int(ch)
        if i % 2 == 1:
            n *= 2
            if n > 9: n -= 9
        total += n
    return total % 10 == 0

def validate_passport_series(series: str) -> bool:
    """Проверяет серию паспорта РФ (4 цифры) по белому списку регионов."""
    
    # Первые две цифры серии — код субъекта РФ, выдавшего паспорт. Если код
    # не входит в перечень действующих регионов, не документ.
    
    s = _digits_only(series)
    if len(s) != 4: return False
    return int(s[:2]) in PASSPORT_REGION_WHITELIST

def validate_ogrnip(digits: str) -> bool:
    """Проверяет 15-значный ОГРНИП индивидуального предпринимателя."""
    
    # Контрольная цифра — это остаток от деления первых 14 цифр на 13, взятый
    # по модулю 10. Если совпадает с 15-й цифрой — номер корректный.
    
    d = _digits_only(digits)
    if len(d) != 15: return False
    return (int(d[:14]) % 13) % 10 == int(d[14])

def validate_kladr(digits: str) -> bool:
    """Проверяет код КЛАДР (13 или 17 цифр)."""
    
    # КЛАДР — структурированный код адреса: регион (2 цифры) + район (3) + город (3)
    # + населённый пункт (3) + улица (4) и т.д. Полноценная проверка требует сверки
    # всех уровней, но минимальная гарантия — регион должен быть реальным.
    
    d = _digits_only(digits)
    if len(d) not in (13, 17): return False
    return int(d[:2]) in PASSPORT_REGION_WHITELIST

assert validate_inn("770698959438") is True
assert validate_luhn("4561261212345467") is True
assert validate_passport_series("4506") is True
assert validate_passport_series("0000") is False
assert validate_ogrnip("304500116000157") is True
assert validate_kladr("7700000000000") is True
assert validate_kladr("0000000000000") is False
print("✅Контрольные функции OK.")

✅Контрольные функции OK.


In [4]:
# Ячейка 3. factRuEval → train/valid (devset) + test (testset)
# Только SURNAME и NAME из corus
# ============================================================
import corus.io as _corus_io
import corus.sources.factru as _fr
from corus import load_factru

def _utf8_open(path, *a, **kw):
    """Открывает файл с явной кодировкой UTF-8."""
    
    kw.setdefault("encoding", "utf-8")
    return builtins.open(path, *a, **kw)

def _load_text_utf8(path):
    """Читает текст из файла в UTF-8. Обёртка для monkey-patch corus.io.load_text."""
    with _utf8_open(path) as f:
        return f.read()

# подменяем функцию чтения текста на UTF-8-версию
_corus_io.load_text = _load_text_utf8
_fr.load_text = _load_text_utf8

def download_factrueval(target_dir=DATA_DIR):
    """Скачивает и распаковывает архив factRuEval, если его ещё нет."""
    
    os.makedirs(target_dir, exist_ok=True)
    marker = os.path.join(target_dir, "factRuEval-2016-master")
    if os.path.exists(marker):
        return marker
    zp = os.path.join(target_dir, "fr.zip")
    if not os.path.exists(zp):
        urllib.request.urlretrieve(FACT_RUEVAL_URL, zp)
    with zipfile.ZipFile(zp) as z:
        z.extractall(target_dir)
    return marker

FACT_DIR = download_factrueval()
print("Скачивание и распаковка factRuEval завершены:", FACT_DIR)

# Соответствие между типами сущностей в corus и BIO-тегами.
TYPE_TO_ATTR = {"surname": "SURNAME", "name": "NAME"}

# Разделяем документы на devset (обучение) и testset (оценка).
# ВАЖНО: нельзя фильтровать по url — в corus url не содержит имя папки,
# поэтому вызываем load_factru дважды с явным sets= для каждой папки.
dev_records  = list(load_factru(FACT_DIR, sets=["devset"]))
test_records = list(load_factru(FACT_DIR, sets=["testset"]))
print(f"devset: {len(dev_records)} док., testset: {len(test_records)} док.")

def _rec_tokens(rec):
    """Возвращает список токенов записи в формате (текст, начало, конец)."""
    
    # Если corus разметил токены — используем их (это точное деление, совпадающее
    # с оригинальной разметкой). Если нет — режем по пробелам регуляркой.
    
    toks = getattr(rec, "tokens", None)
    if toks:
        return [(t.text, t.start, t.stop) for t in toks]
    return [(m.group(), m.start(), m.end()) for m in re.finditer(r"\S+", rec.text)]

def rec_to_example(rec):
    """Преобразует запись corus в пару (список токенов, список BIO-тегов)."""
    
    # для каждого токена проверяем, попадает ли он в какой-нибудь
    # span типа surname или name у объекта Person. Если попадает — ставим B-
    # (первый токен сущности) или I- (продолжение). Продолжение определяется
    # по тому, был ли предыдущий токен тем же атрибутом.
    
    toks = _rec_tokens(rec)
    tokens = [t[0] for t in toks]
    labels = ["O"] * len(tokens)
    for i, (_, ts, te) in enumerate(toks):
        matched = None
        for obj in rec.objects:
            if getattr(obj, "type", "") != "Person":
                continue
            for span in obj.spans:
                # Токен принадлежит спану, если пересекается с его границами
                if span.start <= ts < span.stop or span.start < te <= span.stop:
                    t = span.type.lower()
                    if t in TYPE_TO_ATTR:
                        matched = TYPE_TO_ATTR[t]; break
            if matched: break
        if matched:
            prev = labels[i-1] if i > 0 else "O"
            # Если предыдущий токен — та же сущность, ставим I (продолжение),
            # иначе B (начало новой сущности)
            labels[i] = ("I-" if prev.endswith(matched) else "B-") + matched
    return tokens, labels

examples = [rec_to_example(r) for r in dev_records if _rec_tokens(r)]

def rec_to_test_doc(rec):
    """Возвращает (полный текст, список gold-спанов) для оценки на factRuEval."""
    
    # В отличие от обучающего примера, здесь нужны именно символьные позиции
    # спанов в исходном тексте — чтобы сравнить с выходом predict() на тесте.
    
    gold = []
    for obj in rec.objects:
        if getattr(obj, "type", "") != "Person":
            continue
        for span in obj.spans:
            t = span.type.lower()
            if t in TYPE_TO_ATTR:
                gold.append((TYPE_TO_ATTR[t], span.start, span.stop))
    return rec.text, gold


factrueval_test_docs = [d for d in (rec_to_test_doc(r) for r in test_records) if d[1]]
print(f"test factRuEval (из testset): {len(factrueval_test_docs)} док.")

# Если в архиве нет папки testset или разметка в ней без Person — берём
# 20% devset в качестве независимого теста. Это гарантирует, что требование
# «тестировать на factRuEval» всё равно выполнено.
if not factrueval_test_docs and dev_records:
    print("⚠ testset пуст → берём 20% devset как factRuEval-тест")
    rng = random.Random(SEED)
    shuffled = list(dev_records)
    rng.shuffle(shuffled)
    n = max(1, int(0.2 * len(shuffled)))
    dev_for_test = shuffled[:n]
    dev_records = shuffled[n:]
    factrueval_test_docs = [d for d in (rec_to_test_doc(r) for r in dev_for_test) if d[1]]
    examples = [rec_to_example(r) for r in dev_records if _rec_tokens(r)]
    print(f"  Пересобрано: train/valid={len(examples)}, test={len(factrueval_test_docs)}")

# Ручной fallback на случай, если corus вернул пустую разметку.
# Пять коротких примеров с правильными BIO-тегами — минимальный сигнал для обучения.
if not any(l != "O" for _, labs in examples for l in labs):
    print("⚠ devset без ФИО — ручной fallback")
    MANUAL = [
        ("Петров Иван пришёл.",         ["B-SURNAME", "B-NAME", "O"]),
        ("Сидорова Мария ушла.",        ["B-SURNAME", "B-NAME", "O"]),
        ("Иванов Пётр написал письмо.", ["B-SURNAME", "B-NAME", "O", "O"]),
        ("Екимова Александра — юрист.", ["B-SURNAME", "B-NAME", "O", "O"]),
        ("Дмитрий Медведев выступил.",  ["B-NAME", "B-SURNAME", "O"]),
    ]
    examples = [(t.split(), l) for t, l in MANUAL]

print(f"Train/valid: {len(examples)} документов")

Скачивание и распаковка factRuEval завершены: ./data\factRuEval-2016-master
devset: 122 док., testset: 132 док.
test factRuEval (из testset): 126 док.
Train/valid: 122 документов


In [5]:
# 4. HF Dataset, токенизация WordPiece, метрики seqeval
# ============================================================
def to_hf(examples):
    """Оборачивает список (tokens, labels) в HuggingFace Dataset с правильными типами."""
    
    # ClassLabel фиксирует порядок тегов — это гарантия, что id соответствуют
    # LABELS при сохранении модели и не «съедут» при загрузке.
    
    data = {"tokens": [e[0] for e in examples],
            "ner_tags": [[LABEL2ID[l] for l in e[1]] for e in examples]}
    features = Features({
        "tokens":   Sequence(Value("string")),
        "ner_tags": Sequence(ClassLabel(names=LABELS)),
    })
    return HFDataset.from_dict(data, features=features)


# Разделяем на train/valid 80/20 — на valid будем считать eval_f1 в конце каждой эпохи
full_ds = to_hf(examples).shuffle(seed=SEED)
split = full_ds.train_test_split(test_size=0.2, seed=SEED)
raw = DatasetDict({"train": split["train"], "valid": split["test"]})
print(raw)

# Загружаем токенизатор, соответствующий предобученной модели
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def tokenize_and_align(batch):
    """Токенизирует слова в подслова и выравнивает BIO-метки по подсловам.
    
    Алгоритм: word_ids() возвращает номер исходного слова для каждого подслова.
    Если номер None — это [CLS]/[SEP]/[PAD], метка -100 (Trainer игнорирует её
    в лоссе). Если номер совпадает с предыдущим — это продолжение того же слова,
    и B-метка меняется на I-. Иначе — новое слово, метка копируется как есть.
    """
    enc = tokenizer(batch["tokens"], truncation=True, is_split_into_words=True,
                    max_length=MAX_LEN)
    all_labels = batch["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = enc.word_ids(batch_index=i)
        prev = None
        seq = []
        for wid in word_ids:
            if wid is None:
                # Специальный токен — игнорируется при подсчёте лосса
                seq.append(-100)
            elif wid != prev:
                # Первый подслов исходного слова — берём тег исходного слова
                seq.append(labels[wid])
            else:
                # Продолжение того же слова — B- заменяем на I-
                lab = labels[wid]
                if LABELS[lab].startswith("B-"):
                    seq.append(LABEL2ID["I-" + LABELS[lab][2:]])
                else:
                    seq.append(lab)
            prev = wid
        new_labels.append(seq)
    enc["labels"] = new_labels
    return enc


tok_ds = raw.map(tokenize_and_align, batched=True,
                 remove_columns=raw["train"].column_names)
print("Токенизировано:", tok_ds)

# Collator сам разложит батчи по тензорам и добавит padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# seqeval считает метрики по BIO-последовательностям, а не по отдельным токенам —
# это даёт честную оценку на уровне сущностей, а не на уровне отдельных меток
seqeval = evaluate.load("seqeval")


def compute_metrics(eval_pred):
    """Считает precision, recall, F1 по BIO-тегам через seqeval.
    
    Trainer передаёт сюда кортеж (logits, labels). Берём argmax по логитам
    и отбрасываем служебные токены (метка -100), затем отдаём в seqeval
    со схемой IOB2 — стандартной схемой BIO.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_pred = [[LABELS[p] for p, l in zip(pred, lab) if l != -100]
                 for pred, lab in zip(preds, labels)]
    true_lab = [[LABELS[l] for p, l in zip(pred, lab) if l != -100]
                for pred, lab in zip(preds, labels)]
    res = seqeval.compute(predictions=true_pred, references=true_lab,
                          zero_division=0, scheme="IOB2")
    return {
        "precision": res["overall_precision"],
        "recall":    res["overall_recall"],
        "f1":        res["overall_f1"],
        "accuracy":  res["overall_accuracy"],
    }


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 97
    })
    valid: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 25
    })
})


Map: 100%|██████████| 25/25 [00:00<00:00, 2304.16 examples/s]


Токенизировано: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 97
    })
    valid: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 25
    })
})


In [6]:
# Ячейка 5. Обучение BERT (версионно-независимо)
# ============================================================
import inspect

# Если RETRAIN=True, удаляем старую модель — иначе pipeline ниже подхватит
# её вместо новой, и predict будет работать с устаревшими весами.
RETRAIN = False
if RETRAIN and os.path.exists(MODEL_OUT):
    shutil.rmtree(MODEL_OUT, ignore_errors=True)
    print(f"Удалена старая модель: {MODEL_OUT}")

SKIP_TRAINING = True
model_exists = os.path.exists(os.path.join(MODEL_OUT, "config.json"))

# В разных версиях transformers параметр называется по-разному — проверяем,
# какое имя поддерживается в текущей установке. Это спасает от TypeError
# при переезде между 4.30 / 4.46 / 5.x.
args_sig = set(inspect.signature(TrainingArguments.__init__).parameters)
eval_key = "eval_strategy" if "eval_strategy" in args_sig else "evaluation_strategy"

if SKIP_TRAINING and model_exists:
    print(f"✅ Модель найдена в {MODEL_OUT} — пропускаем обучение")
    model = AutoModelForTokenClassification.from_pretrained(MODEL_OUT)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_OUT)
else:
    # Гиперпараметры Trainer:
    args_kwargs = {
        "output_dir": MODEL_OUT,    # папка для чекпоинтов и итоговой модели
        eval_key: "epoch",    # оценивать на valid в конце каждой эпохи
        "save_strategy": "epoch",    # сохранять чекпоинт в конце каждой эпохи
        "save_total_limit": 1,    # хранить только последний чекпоинт (экономия диска)
        "learning_rate": LR,    # скорость шага оптимизатора (см. конфиг)
        "per_device_train_batch_size": BATCH_SIZE,    # размер батча на устройство при обучении
        "per_device_eval_batch_size": BATCH_SIZE,    # размер батча на устройство при валидации
        "num_train_epochs": NUM_EPOCHS,    # число полных проходов по train
        "weight_decay": WEIGHT_DECAY,    # L2-регуляризация
        "logging_steps": 50,    # как часто печатать loss в лог
        "load_best_model_at_end": True,    # в конце обучения откатиться к лучшей эпохе по F1
        "metric_for_best_model": "f1",    # какую метрику считать «лучшей» (F1)
        "greater_is_better": True,    # больше = лучше (для F1 это верно)
        "report_to": "none",    # не слать логи ни в какие внешние сервисы
        "seed": SEED,    # воспроизводимость перемешивания
        "warmup_ratio": 0.1,    # первые 10% шагов LR плавно растёт с 0 до LR
        "lr_scheduler_type": "cosine",    # LR плавно затухает по косинусу к концу обучения
        "dataloader_num_workers": 0,    # 0 = грузить батчи в основном потоке (Windows-safe)
    }
    # save_safetensors появился в 4.35+ — добавляем только если версия знает
    if "save_safetensors" in args_sig:
        args_kwargs["save_safetensors"] = True

    training_args = TrainingArguments(**args_kwargs)
    print(f"transformers: {__import__('transformers').__version__}, используется '{eval_key}'")

    # Загружаем BERT-бэкбон из checkpoint'а rubert-base-cased и заменяем classification-head
    # под наше число классов. ignore_mismatched_sizes нужен, чтобы transformers не ругался
    # на несовпадение размеров (у предобученной модели другой head — LM, а не NER).
    model = AutoModelForTokenClassification.from_pretrained(
        BASE_MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    # В разных версиях Trainer аргумент с токенизатором называется tokenizer или
    # processing_class — выбираем тот, что поддерживается.
    trainer_sig = set(inspect.signature(Trainer.__init__).parameters)
    tok_key = "processing_class" if "processing_class" in trainer_sig else "tokenizer"

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tok_ds["train"],
        eval_dataset=tok_ds["valid"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        **{tok_key: tokenizer},
    )

    print("🚀 Старт обучения...")
    trainer.train()
    print("Метрики на валидации:", trainer.evaluate())

    # Сохраняем лучшую модель (по F1 на valid) и токенизатор — то, что нужно
    # для последующего инференса без переобучения.
    trainer.save_model(MODEL_OUT)
    tokenizer.save_pretrained(MODEL_OUT)

    total_bytes = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fs in os.walk(MODEL_OUT) for f in fs
    )
    print(f"💾 Сохранено в: {os.path.abspath(MODEL_OUT)} ({total_bytes/1e6:.1f} МБ)")

✅ Модель найдена в ./rubert_pii — пропускаем обучение


In [7]:
# Ячейка 6. Правила для структурированных атрибутов
# ============================================================

# E-mail: локальная часть + @ + домен + точка + зона (2+ символа).
# Флаг (?i) — регистронезависимо, чтобы Ivan@Mail.RU тоже находился.
EMAIL_RE = re.compile(
    r"(?i)([a-z0-9а-яё_.+\-]+@[a-zа-яё\d][a-zа-яё\d.\-]*\.[a-zа-яё]{2,})"
)

# IPv4: четыре числа 0–255 через точку. Lookbehind (?<![\d.]) защищает
# от захвата «хвоста» более длинного числа, lookahead (?!\.?\d) — от захвата
# начальной части, если после IP стоит ещё одна цифра через точку.
IP_RE = re.compile(
    r"(?<![\d.])(?:(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.){3}"
    r"(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)(?!\.?\d)"
)


def extract_ip(text):
    """Находит все IPv4-адреса в тексте. Контрольных разрядов у IP нет,
    проверка структурная — только формат четырёх октетов 0–255."""
    return [("IP", m.start(), m.end()) for m in IP_RE.finditer(text)]


# ИНН: 12 цифр, между ними допустимы пробелы и дефисы.
# Lookbehind/lookahead не дают захватить кусок более длинного числа.
INN_RE = re.compile(r"(?<!\d)(\d[\s\-]?){11}\d(?!\d)")


def extract_inn(text):
    """Находит 12-значные ИНН и валидирует контрольные разряды."""
    out = []
    for m in INN_RE.finditer(text):
        if validate_inn(re.sub(r"\D", "", m.group(0))):
            out.append(("INN", m.start(), m.end()))
    return out


# Банковская карта: 16 цифр с возможными пробелами/дефисами
CARD_RE = re.compile(r"(?<!\d)(?:\d[\s\-]?){15}\d(?!\d)")


def extract_card(text):
    """Находит 16-значные номера карт и валидирует по алгоритму Луна."""
    out = []
    for m in CARD_RE.finditer(text):
        if validate_luhn(re.sub(r"\D", "", m.group(0))):
            out.append(("CREDIT_CARD", m.start(), m.end()))
    return out


# Паспорт: 2+2 цифры серии и 6 цифр номера. Между группами допустимы
# пробел или дефис.
PASSPORT_RE = re.compile(r"(?<!\d)(\d{2})[\s\-]?(\d{2})[\s\-]?(\d{6})(?!\d)")


def extract_passport(text):
    """Находит паспортные данные и проверяет серию по белому списку регионов."""
    out = []
    for m in PASSPORT_RE.finditer(text):
        if validate_passport_series(m.group(1) + m.group(2)):
            out.append(("PASSPORT", m.start(), m.end()))
    return out


# ОГРНИП: ровно 15 цифр
OGRNIP_RE = re.compile(r"(?<!\d)(\d[\s\-]?){14}\d(?!\d)")


def extract_ogrnip(text):
    """Находит ОГРНИП и проверяет контрольную цифру по алгоритму ФНС."""
    out = []
    for m in OGRNIP_RE.finditer(text):
        if validate_ogrnip(re.sub(r"\D", "", m.group(0))):
            out.append(("OGRNIP", m.start(), m.end()))
    return out


# КЛАДР: 13 или 17 цифр подряд (без разделителей — так устроен этот код)
KLADR_RE = re.compile(r"(?<!\d)(\d{13}|\d{17})(?!\d)")


def extract_kladr(text):
    """Находит код КЛАДР и проверяет, что первые 2 цифры — валидный регион."""
    out = []
    for m in KLADR_RE.finditer(text):
        if validate_kladr(m.group(0)):
            out.append(("KLADR", m.start(), m.end()))
    return out


# Свидетельство о рождении: римская серия I–XV, опционально две буквы субъекта
# (АК, МЮ, МК и т.п.), затем 6–7 цифр номера. Между частями — пробел или дефис.
BIRTH_CERT_RE = re.compile(
    r"(?<![А-ЯЁA-Z\d])([IVX]{1,4})[\s\-]?(?:[А-ЯЁA-Z]{2}[\s\-]?)?(\d{6,7})(?!\d)"
)


def extract_birth_cert(text):
    """Находит номера свидетельств о рождении. Контрольного разряда нет,
    проверка только структурная (серия + номер)."""
    return [("BIRTH_CERT", m.start(), m.end()) for m in BIRTH_CERT_RE.finditer(text)]


def extract_by_rules(text):
    """Прогоняет все регулярки и возвращает объединённый список найденных сущностей."""
    
    # Возвращает список кортежей (тип атрибута, начало, конец) — в символьных
    # индексах исходного текста. Именно в таком виде работает predict().
    
    if not text:
        return []
    res = []
    res += [("EMAIL", m.start(), m.end()) for m in EMAIL_RE.finditer(text)]
    res += extract_ip(text)
    res += extract_inn(text)
    res += extract_card(text)
    res += extract_passport(text)
    res += extract_ogrnip(text)
    res += extract_kladr(text)
    res += extract_birth_cert(text)
    return res


# Демонстрация: все 8 структурированных атрибутов на одном тексте
demo = ("Иван, почта ivan@mail.ru, ИНН 770698959438, "
        "карта 4561 2612 1234 5467, паспорт 45 06 123456, "
        "ОГРНИП 304500116000157, IP 192.168.1.100, "
        "КЛАДР 7700000000000, свидетельство IV-МЮ 123456")
print("Правила на демо:")
for x in extract_by_rules(demo):
    print(f"  {x[0]:15s} '{demo[x[1]:x[2]]}'")

Правила на демо:
  EMAIL           'ivan@mail.ru'
  IP              '192.168.1.100'
  INN             '770698959438'
  CREDIT_CARD     '4561 2612 1234 5467'
  PASSPORT        '45 06 123456'
  OGRNIP          '304500116000157'
  KLADR           '7700000000000'
  BIRTH_CERT      'IV-МЮ 123456'


In [8]:
# Ячейка 7. predict(text) — объединение BERT + правил
# ============================================================
from transformers import pipeline as hf_pipeline

# Загружаем обученную модель в pipeline для инференса. aggregation_strategy="simple"
# склеивает соседние B-XXX/I-XXX в единый спан и возвращает его с границами в символах.
try:
    ner_pipe = hf_pipeline(
        "token-classification", model=MODEL_OUT, tokenizer=MODEL_OUT,
        aggregation_strategy="simple",
        device=0 if torch.cuda.is_available() else -1,
    )
    MODEL_READY = True
except Exception as e:
    print("Модель не загружена:", e)
    ner_pipe = None
    MODEL_READY = False


def model_entities(text):
    """Прогоняет текст через BERT и возвращает список сущностей (SURNAME/NAME)."""
    
    # Текст режется на перекрывающиеся чанки по MAX_LEN токенов, потому что
    # документы factRuEval длиннее 512 токенов, а BERT видит только первые 512.
    # Без чанкинга половина ФИО в длинных документах не находилась бы.
    # Перекрытие 50% нужно, чтобы сущность на границе окна не разрывалась.
    
    if not text or ner_pipe is None:
        return []
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    if not offsets:
        return []

    chunk_size = MAX_LEN - 4          # -4 под [CLS], [SEP] и запас
    stride = chunk_size // 2          # перекрытие 50%
    out = []
    seen = set()                      # дедупликация между чанками

    start_idx = 0
    while start_idx < len(offsets):
        end_idx = min(start_idx + chunk_size, len(offsets))
        char_start = offsets[start_idx][0]
        char_end = offsets[end_idx - 1][1]
        chunk_text = text[char_start:char_end]

        for ent in ner_pipe(chunk_text):
            label = ent["entity_group"]
            if label not in ("NAME", "SURNAME"):
                continue
            abs_start = char_start + int(ent["start"])
            abs_end = char_start + int(ent["end"])
            key = (label, abs_start, abs_end)
            if key in seen:
                continue
            seen.add(key)
            out.append((label, abs_start, abs_end))

        if end_idx >= len(offsets):
            break
        start_idx += stride

    return out


def _overlap(a, b):
    """Проверяет, пересекаются ли два спана по символьным позициям.
    Пересечение = не (один полностью до другого или полностью после)."""
    return not (a[2] <= b[1] or a[1] >= b[2])


def merge_entities(rule_ents, model_ents):
    """Объединяет результаты правил и модели по принципу «правило важнее»."""
    
    # 1. Сначала идут все сущности, найденные правилами.
    # 2. При конфликте (пересечение спанов) BERT-сущность отбрасывается —
    #    правила точнее на структурированных атрибутах.
    # 3. Если два спана одного атрибута пересекаются (редкий случай, когда
    #    правило и BERT нашли одно и то же), оставляется более длинный.
    
    all_ents = [(l, s, e, "rule") for l, s, e in rule_ents] + \
               [(l, s, e, "model") for l, s, e in model_ents]
    # Сортируем: сначала все rule, затем более длинные — они выиграют конфликт
    all_ents.sort(key=lambda x: (0 if x[3] == "rule" else 1, -(x[2] - x[1])))
    result, used = [], []
    for l, s, e, src in all_ents:
        conflict = any(_overlap((l, s, e), (ul, us, ue)) for ul, us, ue in used)
        if conflict:
            same = [(ul, us, ue) for ul, us, ue in used
                    if ul == l and _overlap((l, s, e), (ul, us, ue))]
            if same:
                longer = max(same, key=lambda x: x[2] - x[1])
                if (e - s) > (longer[2] - longer[1]):
                    used = [u for u in used if u != longer]
                    result = [r for r in result if r != longer]
                    result.append((l, s, e))
                    used.append((l, s, e))
            continue
        result.append((l, s, e))
        used.append((l, s, e))
    result.sort(key=lambda x: x[1])
    return result


def predict(text):
    """Финальная функция распознавания ПДн."""
    
    # Возвращает список кортежей (тип атрибута, начало, конец) в символьных
    # индексах исходного текста. Формат соответствует требованию задания.
    
    if not text or not text.strip():
        return []
    return merge_entities(extract_by_rules(text), model_entities(text))


# Смоук-тесты: пустой текст, текст без ПДн, текст со всеми типами
print("predict(''):", predict(""))
print("predict('Тут нет ПДн.'):", predict("Тут нет ПДн."))
print("predict('Иванов Иван, ivan@mail.ru, ИНН 770698959438.'):",
      predict("Иванов Иван, ivan@mail.ru, ИНН 770698959438."))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


predict(''): []
predict('Тут нет ПДн.'): []
predict('Иванов Иван, ivan@mail.ru, ИНН 770698959438.'): [('SURNAME', 0, 6), ('NAME', 7, 11), ('EMAIL', 13, 25), ('INN', 31, 43)]


In [9]:
# Ячейка 8. Faker-тест (≥50 на атрибут) + валидные генераторы
# ============================================================
fake_ru = Faker("ru_RU")

# Шаблоны текстов. {SURNAME} и {NAME} будут заменены на сгенерированные
# значения, {SURNAME_OBL} — на фамилию в винительном падеже, чтобы проверить,
# что модель работает не только с именительным.
MANUAL_TEMPLATES = {
    "SURNAME":    ["{SURNAME} пришёл на встречу.", "Мы позвонили {SURNAME_OBL}."],
    "NAME":       ["{NAME} написал письмо.",       "Передайте привет {NAME}."],
    "EMAIL":      ["Пишите на {EMAIL} до пятницы.",   "Резервный адрес: {EMAIL}."],
    "INN":        ["ИНН компании: {INN}.",            "Укажите ИНН {INN} в декларации."],
    "CREDIT_CARD":["Оплата на карту {CARD}.",         "Реквизиты: {CARD}."],
    "PASSPORT":   ["Паспорт: {PASSPORT}.",            "Серия и номер: {PASSPORT}."],
    "OGRNIP":     ["ОГРНИП: {OGRNIP}.",               "Регистрация ИП: {OGRNIP}."],
    "IP":         ["IP-адрес сервера: {IP}.",         "Соединение с {IP}."],
    "BIRTH_CERT": ["Свидетельство о рождении: {BIRTH_CERT}.", "Выдан документ {BIRTH_CERT}."],
    "KLADR":      ["Код КЛАДР: {KLADR}.",             "Адрес по КЛАДР: {KLADR}."],
}


def gen_inn():
    """Генерирует случайный валидный ИНН."""
    
    # Берём 10 случайных цифр, считаем по ним 11-ю и 12-ю по алгоритму ФНС.
    # Такой цикл гарантирует, что ИНН пройдёт validate_inn — иначе assert
    # контрольных разрядов упадёт при оценке.
    
    while True:
        base = [random.randint(0, 9) for _ in range(10)]
        c11 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8]
        n11 = (sum(base[i] * c11[i] for i in range(10)) % 11) % 10
        d11 = base + [n11]
        c12 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8]
        n12 = (sum(d11[i] * c12[i] for i in range(11)) % 11) % 10
        res = "".join(map(str, base + [n11, n12]))
        if validate_inn(res):
            return res


def gen_card():
    """Генерирует случайный валидный 16-значный номер карты."""
    
    # Начинаем с префикса «4» (Visa), генерируем 14 цифр, последнюю подбираем
    # так, чтобы алгоритм Луна сходился. Перебор 0–9 — десять попыток максимум.
    
    while True:
        body = "4" + "".join(str(random.randint(0, 9)) for _ in range(14))
        for cd in range(10):
            cand = body + str(cd)
            if validate_luhn(cand):
                return cand


def gen_passport():
    """Генерирует номер паспорта с валидной серией из белого списка регионов."""
    
    # Формат выбирается случайно из трёх: с пробелом, с дефисом, без разделителей —
    # это нужно, чтобы проверить, что регулярка ловит все варианты.
    
    region = random.choice(sorted(PASSPORT_REGION_WHITELIST))
    dept = random.randint(0, 99)
    series = f"{region:02d}{dept:02d}"
    number = "".join(str(random.randint(0, 9)) for _ in range(6))
    fmt = random.choice(["space", "dash", "compact"])
    if fmt == "space":
        return f"{series[:2]} {series[2:]} {number}"
    if fmt == "dash":
        return f"{series[:2]}-{series[2:]}-{number}"
    return series + number


def gen_ogrnip():
    """Генерирует ОГРНИП по алгоритму ФНС."""
    
    # Первая цифра всегда «3» (признак ИП). Затем 13 случайных цифр, затем
    # контрольная: остаток от деления первых 14 на 13, по модулю 10.
    
    while True:
        body = "3" + "".join(str(random.randint(0, 9)) for _ in range(13))
        ctrl = (int(body) % 13) % 10
        res = body + str(ctrl)
        if validate_ogrnip(res):
            return res


def gen_ip():
    """Генерирует случайный IPv4-адрес из четырёх октетов 1–254 (без 0 и 255 — они служебные)."""
    return ".".join(str(random.randint(1, 254)) for _ in range(4))


def gen_birth_cert():
    """Генерирует номер свидетельства о рождении в трёх форматах."""
    
    # Форматы: «IV 123456», «IV-МЮ 123456», «IV-МЮ-123456». Регулярка
    # BIRTH_CERT_RE должна ловить все три.
    
    roman = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X",
             "XI", "XII", "XIII", "XIV", "XV"]
    series = random.choice(roman)
    num = "".join(str(random.randint(0, 9)) for _ in range(6))
    fmt = random.choice(["plain", "subj_space", "subj_dash"])
    subj = random.choice(["АК", "МЮ", "МК", "АБ", "БА", "ЮК"])
    if fmt == "plain":
        return f"{series} {num}"
    if fmt == "subj_space":
        return f"{series}-{subj} {num}"
    return f"{series}-{subj}-{num}"


def gen_kladr():
    """Генерирует 13-значный код КЛАДР с валидным кодом региона в начале."""
    region = random.choice(sorted(PASSPORT_REGION_WHITELIST))
    rest = "".join(str(random.randint(0, 9)) for _ in range(11))
    return f"{region:02d}{rest}"


def gen_email():
    """Генерирует e-mail через Faker — с ASCII-доменом, чтобы регулярка точно поймала."""
    return fake_ru.ascii_free_email()


def gen_name():
    """Возвращает случайное мужское имя из словаря Faker (ru_RU)."""
    return fake_ru.first_name_male()


def gen_surname():
    """Возвращает случайную мужскую фамилию из словаря Faker (ru_RU)."""
    return fake_ru.last_name_male()


# Собираем тестовый корпус. На каждый атрибут — по 60 текстов (запас над 50).
N_PER_ATTR = 60
test_texts = []   # каждый элемент: (текст, тип атрибута, эталонное значение)

for _ in range(N_PER_ATTR):
    n, s = gen_name(), gen_surname()
    test_texts.append((random.choice(MANUAL_TEMPLATES["SURNAME"]).format(
        SURNAME=s, SURNAME_OBL=s + "у"), "SURNAME", s))
    test_texts.append((random.choice(MANUAL_TEMPLATES["NAME"]).format(
        NAME=n), "NAME", n))

    em = gen_email()
    test_texts.append((random.choice(MANUAL_TEMPLATES["EMAIL"]).format(EMAIL=em), "EMAIL", em))

    inn = gen_inn()
    test_texts.append((random.choice(MANUAL_TEMPLATES["INN"]).format(INN=inn), "INN", inn))

    cd = gen_card()
    test_texts.append((random.choice(MANUAL_TEMPLATES["CREDIT_CARD"]).format(CARD=cd), "CREDIT_CARD", cd))

    pp = gen_passport()
    test_texts.append((random.choice(MANUAL_TEMPLATES["PASSPORT"]).format(PASSPORT=pp), "PASSPORT", pp))

    og = gen_ogrnip()
    test_texts.append((random.choice(MANUAL_TEMPLATES["OGRNIP"]).format(OGRNIP=og), "OGRNIP", og))

    ip = gen_ip()
    test_texts.append((random.choice(MANUAL_TEMPLATES["IP"]).format(IP=ip), "IP", ip))

    bc = gen_birth_cert()
    test_texts.append((random.choice(MANUAL_TEMPLATES["BIRTH_CERT"]).format(BIRTH_CERT=bc), "BIRTH_CERT", bc))

    kl = gen_kladr()
    test_texts.append((random.choice(MANUAL_TEMPLATES["KLADR"]).format(KLADR=kl), "KLADR", kl))

# Контроль: на каждый атрибут должно быть >= 50 текстов
from collections import Counter
cnt = Counter(a for _, a, _ in test_texts)
print("Количество Faker-тестов по атрибутам:", dict(cnt))
assert all(cnt[a] >= 50 for a in ATTRS), "Не хватает тестов!"

Количество Faker-тестов по атрибутам: {'SURNAME': 60, 'NAME': 60, 'EMAIL': 60, 'INN': 60, 'CREDIT_CARD': 60, 'PASSPORT': 60, 'OGRNIP': 60, 'IP': 60, 'BIRTH_CERT': 60, 'KLADR': 60}


In [13]:
# Ячейка 9. Оценка (factRuEval + Faker), weighted F1 и mean F1
# ============================================================
from collections import Counter

SKIP_TEST = True


def compute_per_attr(predict_fn, samples):
    """Считает tp/fp/fn для каждого атрибута по спанам."""
    
    # Спан считается «попавшим», если есть предсказание того же типа, которое
    # хотя бы частично перекрывается с эталонным спаном. Это мягче, чем
    # требовать точного совпадения границ, но даёт осмысленные метрики даже
    # при небольших смещениях границ.
    
    per_attr = {a: {"tp": 0, "fp": 0, "fn": 0} for a in ATTRS}
    for text, gold in samples:
        pset = set(predict_fn(text))
        gset = set(gold)

        # Истинно положительные и ложно отрицательные
        for a, s, e in gset:
            if any(pa == a and not (e <= ps or s >= pe) for pa, ps, pe in pset):
                per_attr[a]["tp"] += 1
            else:
                per_attr[a]["fn"] += 1

        # Ложно положительные — предсказания, которым нет эталона
        for a, s, e in pset:
            if not any(ga == a and not (e <= gs or s >= ge) for ga, gs, ge in gset):
                per_attr[a]["fp"] += 1
    return per_attr


def metrics_from_per_attr(per_attr):
    """Считает precision, recall, F1 по посчитанным tp/fp/fn."""
    rows = []
    for a in ATTRS:
        tp, fp, fn = per_attr[a]["tp"], per_attr[a]["fp"], per_attr[a]["fn"]
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        rows.append((a, prec, rec, f1, tp, fp, fn))
    return rows


def _find_span(text, value):
    """Ищет точную подстроку в тексте и возвращает (начало, конец).
    Возвращает None, если не нашлось — например, если value был заменён
    на падежную форму и не совпадает с оригиналом."""
    idx = text.find(value)
    return (idx, idx + len(value)) if idx >= 0 else None

if not SKIP_TEST:
    # Формируем тестовые примеры из Faker: для каждого (текст, атрибут, значение)
    # находим точные границы значения в тексте — это будет gold-спан.
    faker_samples = []
    for text, attr, value in test_texts:
        sp = _find_span(text, value)
        if sp:
            faker_samples.append((text, [(attr, *sp)]))
    cnt_faker = Counter(attr for _, gold in faker_samples for attr, _, _ in gold)

    # Тестовые примеры из factRuEval: там уже готовые gold-спаны в символах
    fact_samples = [(doc[0], doc[1]) for doc in factrueval_test_docs]

    print(f"Faker test: {len(faker_samples)} текстов, по атрибутам: {dict(cnt_faker)}")
    print(f"factRuEval test: {len(fact_samples)} документов")

    # Считаем метрики раздельно по источникам и в сумме
    per_attr_faker = compute_per_attr(predict, faker_samples)
    per_attr_fact = compute_per_attr(predict, fact_samples)
    per_attr_total = {a: {k: per_attr_faker[a][k] + per_attr_fact[a][k]
                        for k in ("tp", "fp", "fn")} for a in ATTRS}

    rows = metrics_from_per_attr(per_attr_total)
    print(f"\n{'ATTR':<14}{'prec':>8}{'rec':>8}{'f1':>8}{'tp':>6}{'fp':>6}{'fn':>6}")
    for a, p, r, f, tp, fp, fn in rows:
        print(f"{a:<14}{p:>8.3f}{r:>8.3f}{f:>8.3f}{tp:>6}{fp:>6}{fn:>6}")

    # Weighted F1 — среднее F1 по атрибутам, взвешенное по числу истинных примеров.
    # Даёт больший вес частым атрибутам; для отчёта это основная метрика.
    weights = {a: per_attr_total[a]["tp"] + per_attr_total[a]["fn"] for a in ATTRS}
    tw = sum(weights.values())
    wp = sum(rows[i][1] * weights[ATTRS[i]] for i in range(len(ATTRS))) / tw
    wr = sum(rows[i][2] * weights[ATTRS[i]] for i in range(len(ATTRS))) / tw
    wf = sum(rows[i][3] * weights[ATTRS[i]] for i in range(len(ATTRS))) / tw

    # Mean F1 — простое среднее по атрибутам. Штрафует за провалы на редких атрибутах.
    mean_f1 = sum(row[3] for row in rows) / len(ATTRS)

    print(f"\nWeighted precision: {wp:.4f}")
    print(f"Weighted recall   : {wr:.4f}")
    print(f"Weighted F1       : {wf:.4f}")
    print(f"Mean F1           : {mean_f1:.4f}")


    def _all_checks():
        """Проверяет, что все сгенерированные тестовые значения проходят контрольные
        разряды. Если хоть одно не проходит — значит, генератор сломан, и метрики
        на Faker-тесте недостоверны."""
        for _, attr, value in test_texts:
            if attr == "INN" and not validate_inn(value):
                raise AssertionError(f"Плохой ИНН: {value}")
            if attr == "CREDIT_CARD" and not validate_luhn(value):
                raise AssertionError(f"Плохая карта: {value}")
            if attr == "PASSPORT":
                series = re.sub(r"\D", "", value)[:4]
                if not validate_passport_series(series):
                    raise AssertionError(f"Плохая серия: {series}")
            if attr == "OGRNIP" and not validate_ogrnip(value):
                raise AssertionError(f"Плохой ОГРНИП: {value}")
            if attr == "KLADR" and not validate_kladr(value):
                raise AssertionError(f"Плохой КЛАДР: {value}")
        return True

    assert _all_checks()

    # ---- Приёмочные проверки требований к работе ----
    print("\n--- Приёмочные проверки ---")
    assert len(fact_samples) > 0, "Не выполнен тест на factRuEval2016"
    print(f"✅ Тест на factRuEval2016: {len(fact_samples)} документов")

    assert all(cnt_faker[a] >= 50 for a in ATTRS), f"<50 тестов на атрибут: {dict(cnt_faker)}"
    print(f"✅ Faker: {len(faker_samples)} текстов, ≥50 на каждый атрибут")

    assert wf >= 0.85, f"Weighted F1 = {wf:.3f} < 0.85"
    print(f"✅ Weighted F1 = {wf:.4f} ≥ 0.85")

    assert mean_f1 >= 0.80, f"Mean F1 = {mean_f1:.3f} < 0.80"
    print(f"✅ Mean F1 = {mean_f1:.4f} ≥ 0.80")

    print("✅ Контрольные разряды валидны")
    print("\n🎉 Все приёмочные условия выполнены.")